# **Introduction to GANs: Vanilla GAN, Conditional GAN, and WGAN on MNIST**

In this notebook you will learn how to train four types of Generative Adversarial Networks (GANs): a **Vanilla GAN**, a **Conditional GAN**, aand a **Wasserstein GAN (WGAN)**, using the MNIST dataset of handwritten digits.

GANs are a type of neural network architecture that learn to generate new data samples that resemble the training dataset. They consist of two components:
1. **Generator**: Produces synthetic data by transforming random noise.
2. **Discriminator** (or **Critic** in WGAN): Distinguishes between real and fake data samples.

The generator and discriminator are trained in an adversarial setup where the generator aims to "fool" the discriminator by generating realistic data, while the discriminator improves its ability to identify fake data.

---

### **What You Will Do**
1. **Vanilla GAN**:
   - Train a GAN where the generator learns to produce realistic MNIST-like images without label conditioning.
   - Monitor the generator's progress by visualizing generated samples during training.
   - Plot the losses of the generator and discriminator to understand the training dynamics.

2. **Conditional GAN (cGAN)**:
   - Train a GAN that generates MNIST images conditioned on the digit label.
   - This means you will be able to control which digit the generator produces by providing a label (e.g., generate a "3" or a "7").
   - Visualize the generator's output to ensure it produces images matching the requested labels.

3. **Wasserstein GAN (WGAN)**:
   - Train a GAN using the Wasserstein distance (Earth Mover's distance) instead of JS divergence.
   - The discriminator becomes a "critic" that outputs unbounded scores instead of probabilities.
   - Uses weight clipping to enforce the Lipschitz constraint.
   - More stable training and meaningful loss curves compared to Vanilla GAN.

In [ ]:
# Import required libraries
import torch
from torch import nn
from torch.utils.data import DataLoader
import torchvision
from torchvision import transforms
import matplotlib.pyplot as plt
import numpy as np
# Set the device to cuda if available 
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# Load MNIST dataset
batch_size = 128
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])

data = DataLoader(
    torchvision.datasets.MNIST('./data', transform=transform, download=True),
    batch_size=batch_size,
    shuffle=True
)

In [ ]:
# Function to visualize generated images (You should call this every 10 epochs in your 50 epoch training loop)
# Will allow you to see how the generator is performing
def plot_generated_images(images, epoch, title="Generated Images"):
    fig, axes = plt.subplots(5, 5, figsize=(4, 4))
    for i, ax in enumerate(axes.flatten()):
        ax.imshow(images[i].reshape(28, 28), cmap='gray')
        ax.axis('off')
    fig.suptitle(f"{title} - Epoch {epoch}", fontsize=16)
    plt.tight_layout()
    plt.show()

## 1. Vanilla GAN with Jensen-Shannon Loss

In [ ]:
# Define the Generator
# Archetecture: 
# 1. Input layer: 100 nodes
# 2. Hidden layer: 256 nodes
# 3. Hidden layer: 512 nodes
# 4. Output layer: 784 nodes
# 5. Activation function: ReLU
# 6. Output activation function: Tanh
# 7. Output shape: 28x28
class Generator(nn.Module):
    def __init__(self, noise_dim):
        super(Generator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(noise_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, 28 * 28),
            nn.Tanh() 
        )

    def forward(self, x):
        return self.model(x).view(-1, 1, 28, 28)

In [ ]:
# Test the Generator, should output torch.Size([128, 1, 28, 28]) --> (batch_size, channels, height, width)
noise_dim = 100
generator = Generator(noise_dim).to(device)
noise = torch.randn(batch_size, noise_dim).to(device)
generated_images = generator(noise)
print(generated_images.shape)

In [ ]:
# Define the Discriminator
# Archetecture:
# 1. Input layer: 784 nodes
# 2. Hidden layer: 512 nodes
# 3. Hidden layer: 256 nodes
# 4. Output layer: 1 node
# 5. Activation function: LeakyReLU
# 6. Output activation function: Sigmoid
# 7. Output shape: 1
class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(28 * 28, 512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x.view(-1, 28 * 28))

In [ ]:
# Test the Discriminator, should output torch.Size([128, 1]) --> (batch_size, 1)
discriminator = Discriminator().to(device)
output = discriminator(generated_images)
print(output.shape)

In [ ]:
# Create a training function for the Vanilla GAN that returns the losses for the discriminator and generator
def train_vanilla_gan(epochs, noise_dim, lr=2e-4):
    generator = Generator(noise_dim).to(device)
    discriminator = Discriminator().to(device)
    criterion = nn.BCELoss()
    optim_g = torch.optim.Adam(generator.parameters(), lr=lr)
    optim_d = torch.optim.Adam(discriminator.parameters(), lr=lr)

    d_losses, g_losses = [], []
    for epoch in range(epochs):
        for real_images, _ in data:
            real_images = real_images.to(device)
            batch_size = real_images.size(0)

            # Train Discriminator
            noise = torch.randn(batch_size, noise_dim).to(device)
            fake_images = generator(noise)
            real_labels = torch.ones(batch_size, 1).to(device)
            fake_labels = torch.zeros(batch_size, 1).to(device)

            d_real_loss = criterion(discriminator(real_images), real_labels)
            d_fake_loss = criterion(discriminator(fake_images.detach()), fake_labels)
            d_loss = d_real_loss + d_fake_loss
            optim_d.zero_grad()
            d_loss.backward()
            optim_d.step()

            # Train Generator
            g_loss = criterion(discriminator(fake_images), real_labels)
            optim_g.zero_grad()
            g_loss.backward()
            optim_g.step()

            d_losses.append(d_loss.item())
            g_losses.append(g_loss.item())

        if epoch % 10 == 0:
            noise = torch.randn(25, noise_dim).to(device)
            generated_images = generator(noise).detach().cpu().numpy()
            plot_generated_images(generated_images, epoch, title="Vanilla GAN")

    return d_losses, g_losses

In [ ]:
# Train Vanilla GAN
d_losses, g_losses = train_vanilla_gan(epochs=50, noise_dim=100)

In [ ]:
# Function to smooth data using a moving average (Use block as is in your code)
def smooth(x, size):
    return np.convolve(x, np.ones(size)/size, mode='valid')
# Create the plot
fig, ax1 = plt.subplots(figsize=(10, 5))
# Plot Discriminator Loss on the left y-axis
color = 'grey'
ax1.set_xlabel('Iterations')
ax1.set_ylabel('Discriminator Loss', color=color)
ax1.plot(smooth(d_losses, 50), label='Discriminator Loss', color=color, linewidth=.5)
ax1.tick_params(axis='y', labelcolor=color)
ax1.axhline(0.5, color='gray', linestyle='--', linewidth=1)  # Reference line for convergence to 0.5
ax1.set_yticks(np.arange(0, 1.1, 0.1))  # Add subticks for discriminator
# Add Generator Loss on the right y-axis
ax2 = ax1.twinx()  # Create a second y-axis
color = 'black'
ax2.set_ylabel('Generator Loss', color=color)
ax2.plot(smooth(g_losses, 50), label='Generator Loss', color=color, linewidth=.5)
ax2.tick_params(axis='y', labelcolor=color)
# Add a title and legends
fig.suptitle("Vanilla GAN Loss Curves")
fig.tight_layout()  # Adjust layout to prevent overlapping
plt.show()

## 2. Conditional GAN
Modify both the generator and discriminator to handle labels.

In [ ]:
# Generator for Conditional GAN
# Same as Vanilla GAN but with an additional label embedding layer
# Use nn.Embedding to create a label embedding layer
# Each number from 0 to 9 will be embedded into a 10-dimensional vector and concatenated with the noise vector
# The label embedding layer will be concatenated with the noise vector in the generator
class ConditionalGenerator(nn.Module):
    def __init__(self, noise_dim, num_classes):
        super(ConditionalGenerator, self).__init__()
        self.label_embedding = nn.Embedding(num_classes, num_classes) # additonal layer for label embedding, (10x10) lookup table (not a dense layer). Passing a lable like 7 will return the 7th row of the embedding matrix.
        self.model = nn.Sequential(
            nn.Linear(noise_dim + num_classes, 256), # add num_classes to noise_dim
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, 28 * 28),
            nn.Tanh()
        )

    def forward(self, noise, labels):
        # Concatenate noise and label embeddings
        labels = self.label_embedding(labels) # shape: (batch_size, num_classes) --> (128, 10)
        x = torch.cat((noise, labels), dim=1) # shape: (batch_size, noise_dim + num_classes) --> (128, 110)
        return self.model(x).view(-1, 1, 28, 28)

In [ ]:
# Test the Conditional Generator, should output torch.Size([128, 1, 28, 28]) --> (batch_size, channels, height, width)
num_classes = 10
conditional_generator = ConditionalGenerator(noise_dim, num_classes).to(device)
noise = torch.randn(batch_size, noise_dim).to(device) # Random noise shape (128, 100)
labels = torch.randint(0, num_classes, (batch_size,)).to(device) # Random labels should be between 0 and 9 shape (128,)
generated_images = conditional_generator(noise, labels)
print(generated_images.shape)

In [ ]:
# Discriminator for Conditional GAN
# Same as Vanilla GAN but with an additional label embedding layer 
class ConditionalDiscriminator(nn.Module):
    def __init__(self, num_classes):
        super(ConditionalDiscriminator, self).__init__()
        self.label_embedding = nn.Embedding(num_classes, num_classes) # Embedding matrix shape: (num_classes, num_classes), i.e., lookup table
        self.model = nn.Sequential(
            nn.Linear(28 * 28 + num_classes, 512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, images, labels):
        # Flatten images and concatenate with label embeddings
        labels = self.label_embedding(labels)
        x = torch.cat((images.view(images.size(0), -1), labels), dim=1) # shape: (batch_size, 28*28 + num_classes) --> (128, 794)
        return self.model(x)

In [ ]:
# Test the Conditional Discriminator, should output torch.Size([128, 1]) --> (batch_size, 1), model guess for real or fake
conditional_discriminator = ConditionalDiscriminator(num_classes).to(device)
output = conditional_discriminator(generated_images, labels)
print(output.shape)

In [ ]:
# The only difference between the Vanilla GAN and Conditional GAN training loops is the addition of 
# Class labels to the generator and discriminator inputs. (differnet from labels for real and fake images)
def train_cgan(epochs, noise_dim, num_classes, lr=2e-4):
    generator = ConditionalGenerator(noise_dim, num_classes).to(device)
    discriminator = ConditionalDiscriminator(num_classes).to(device)

    criterion = nn.BCELoss()
    optim_g = torch.optim.Adam(generator.parameters(), lr=lr)
    optim_d = torch.optim.Adam(discriminator.parameters(), lr=lr)

    d_losses, g_losses = [], []
    for epoch in range(epochs):
        for real_images, labels in data:
            real_images, labels = real_images.to(device), labels.to(device)
            batch_size = real_images.size(0)

            # Create real and fake labels
            real_targets = torch.ones(batch_size, 1).to(device)
            fake_targets = torch.zeros(batch_size, 1).to(device)

            # Train Discriminator
            noise = torch.randn(batch_size, noise_dim).to(device)
            fake_images = generator(noise, labels)
            real_loss = criterion(discriminator(real_images, labels), real_targets)
            fake_loss = criterion(discriminator(fake_images.detach(), labels), fake_targets)
            d_loss = real_loss + fake_loss
            optim_d.zero_grad()
            d_loss.backward()
            optim_d.step()

            # Train Generator
            noise = torch.randn(batch_size, noise_dim).to(device)
            fake_images = generator(noise, labels)
            g_loss = criterion(discriminator(fake_images, labels), real_targets)
            optim_g.zero_grad()
            g_loss.backward()
            optim_g.step()

            d_losses.append(d_loss.item())
            g_losses.append(g_loss.item())

        # Visualize generated images every 10 epochs
        if epoch % 10 == 0:
            noise = torch.randn(25, noise_dim).to(device)
            sample_labels = torch.randint(0, num_classes, (25,)).to(device)
            generated_images = generator(noise, sample_labels).detach().cpu().numpy()
            plot_generated_images(generated_images, epoch, title=f"Conditional GAN (Epoch {epoch})")

    return d_losses, g_losses, generator

In [ ]:
# Train Conditional GAN
num_classes = 10  # MNIST has 10 classes (0-9)
d_losses, g_losses, generator  = train_cgan(epochs=50, noise_dim=100, num_classes=num_classes)

In [ ]:
# Plot the losses for the Conditional GAN training loop (Use block as is in your code)
fig, ax1 = plt.subplots(figsize=(10, 5))
# Plot Discriminator Loss on the left y-axis
color = 'grey'
ax1.set_xlabel('Iterations')
ax1.set_ylabel('Discriminator Loss', color=color)
ax1.plot(smooth(d_losses, 50), label='Discriminator Loss', color=color, linewidth=.5)
ax1.tick_params(axis='y', labelcolor=color)
# Add Generator Loss on the right y-axis
ax2 = ax1.twinx()  # Create a second y-axis
color = 'black'
ax2.set_ylabel('Generator Loss', color=color)
ax2.plot(smooth(g_losses, 50), label='Generator Loss', color=color, linewidth=.5)
ax2.tick_params(axis='y', labelcolor=color)
# Add a title and legends
fig.suptitle("Conditional GAN Loss Curves")
fig.tight_layout()  # Adjust layout to prevent overlapping
plt.show()

In [ ]:
# Plot Function for Conditional GAN (Use block as is in your code)
def plot_samples_for_condition(generator, condition, noise_dim=100, num_samples=10):
    """
    Generate and plot `num_samples` samples from the generator for a specific condition.

    Args:
        generator: Trained Conditional Generator model.
        condition: Integer (0-9) representing the desired digit class.
        noise_dim: Dimensionality of the noise vector.
        num_samples: Number of samples to generate.
    """
    generator.eval()  # Set generator to evaluation mode
    with torch.no_grad():
        # Create noise vectors
        noise = torch.randn(num_samples, noise_dim).to(device)
        # Create a tensor of repeated condition labels
        labels = torch.full((num_samples,), condition, dtype=torch.long).to(device)
        # Generate images
        generated_images = generator(noise, labels).cpu().numpy()

    # Plot the images in a single row
    fig, axes = plt.subplots(1, num_samples, figsize=(15, 3))
    for i in range(num_samples):
        axes[i].imshow(generated_images[i].reshape(28, 28), cmap='gray')
        axes[i].axis('off')
    plt.suptitle(f"Generated Samples for Condition: {condition}", fontsize=16)
    plt.show()

In [ ]:
# Play with the class condition to see the generated images for each class
# Your model should be able to generate images of the digits 0-9 on demand
plot_samples_for_condition(generator, condition=0, noise_dim=100, num_samples=10)

## 3. Wasserstein GAN (WGAN)

The Wasserstein GAN addresses training instability in Vanilla GANs by using the Wasserstein distance (Earth Mover's distance) instead of the JS divergence. Key differences:

1. **Critic instead of Discriminator**: Outputs a real-valued score (no sigmoid), representing how "real" an image looks.
2. **Wasserstein Loss**: `D(real) - D(fake)` for the critic, `-D(fake)` for the generator.
3. **Weight Clipping**: Enforces the Lipschitz constraint required for the Wasserstein distance.
4. **More critic updates**: Train the critic multiple times per generator update for better gradients.

In [ ]:
# WGAN Generator (same architecture as Vanilla GAN)
# The generator architecture remains unchanged - only the loss function differs
class WGANGenerator(nn.Module):
    def __init__(self, noise_dim):
        super(WGANGenerator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(noise_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, 28 * 28),
            nn.Tanh()
        )

    def forward(self, x):
        return self.model(x).view(-1, 1, 28, 28)

In [ ]:
# WGAN Critic (NOT Discriminator)
# Key difference: NO sigmoid at output - outputs unbounded real-valued score
# The critic learns to output higher scores for real images and lower for fake
class WGANCritic(nn.Module):
    def __init__(self):
        super(WGANCritic, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(28 * 28, 512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 1)  # No sigmoid! Output is an unbounded score
        )

    def forward(self, x):
        return self.model(x.view(-1, 28 * 28))

In [ ]:
# Test WGAN models
wgan_generator = WGANGenerator(noise_dim).to(device)
wgan_critic = WGANCritic().to(device)

noise = torch.randn(batch_size, noise_dim).to(device)
fake_images = wgan_generator(noise)
critic_output = wgan_critic(fake_images)

print(f"Generator output shape: {fake_images.shape}")  # Should be [128, 1, 28, 28]
print(f"Critic output shape: {critic_output.shape}")    # Should be [128, 1]
print(f"Critic output range: [{critic_output.min().item():.3f}, {critic_output.max().item():.3f}]")  # Unbounded!

In [ ]:
# WGAN Training Function
# Key differences from Vanilla GAN:
# 1. Use Wasserstein loss instead of BCE
# 2. Weight clipping to enforce Lipschitz constraint
# 3. Train critic n_critic times per generator update
# 4. Use RMSprop optimizer (works better with weight clipping)
def train_wgan(epochs, noise_dim, n_critic=5, clip_value=0.01, lr=5e-5):
    generator = WGANGenerator(noise_dim).to(device)
    critic = WGANCritic().to(device)
    
    # RMSprop is recommended for WGAN (Adam can cause instability with weight clipping)
    optim_g = torch.optim.RMSprop(generator.parameters(), lr=lr)
    optim_c = torch.optim.RMSprop(critic.parameters(), lr=lr)

    c_losses, g_losses = [], []
    
    for epoch in range(epochs):
        for i, (real_images, _) in enumerate(data):
            real_images = real_images.to(device)
            batch_size = real_images.size(0)

            # ---------------------
            # Train Critic (n_critic times per generator update)
            # ---------------------
            for _ in range(n_critic):
                noise = torch.randn(batch_size, noise_dim).to(device)
                fake_images = generator(noise).detach()
                
                # Wasserstein loss: maximize E[C(real)] - E[C(fake)]
                # Equivalent to minimizing -E[C(real)] + E[C(fake)]
                c_real = critic(real_images).mean()
                c_fake = critic(fake_images).mean()
                c_loss = -c_real + c_fake  # We minimize this
                
                optim_c.zero_grad()
                c_loss.backward()
                optim_c.step()
                
                # Weight clipping to enforce Lipschitz constraint
                for p in critic.parameters():
                    p.data.clamp_(-clip_value, clip_value)

            # ---------------------
            # Train Generator
            # ---------------------
            noise = torch.randn(batch_size, noise_dim).to(device)
            fake_images = generator(noise)
            
            # Generator wants to maximize C(fake), i.e., minimize -C(fake)
            g_loss = -critic(fake_images).mean()
            
            optim_g.zero_grad()
            g_loss.backward()
            optim_g.step()

            c_losses.append(c_loss.item())
            g_losses.append(g_loss.item())

        # Visualize every 10 epochs
        if epoch % 10 == 0:
            noise = torch.randn(25, noise_dim).to(device)
            generated_images = generator(noise).detach().cpu().numpy()
            plot_generated_images(generated_images, epoch, title="WGAN")
            # Print Wasserstein estimate (negative critic loss approximates W-distance)
            print(f"Epoch {epoch}, Wasserstein estimate: {-c_loss.item():.4f}")

    return c_losses, g_losses, generator

In [ ]:
# Train WGAN (may take longer due to multiple critic updates per generator update)
c_losses, g_losses_wgan, wgan_generator = train_wgan(epochs=50, noise_dim=100)

In [ ]:
# Plot WGAN losses
# Note: Unlike Vanilla GAN, the critic loss in WGAN is interpretable!
# The negative critic loss approximates the Wasserstein distance.
fig, ax1 = plt.subplots(figsize=(10, 5))

color = 'grey'
ax1.set_xlabel('Iterations')
ax1.set_ylabel('Critic Loss (≈ -Wasserstein Distance)', color=color)
ax1.plot(smooth(c_losses, 50), label='Critic Loss', color=color, linewidth=.5)
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()
color = 'black'
ax2.set_ylabel('Generator Loss', color=color)
ax2.plot(smooth(g_losses_wgan, 50), label='Generator Loss', color=color, linewidth=.5)
ax2.tick_params(axis='y', labelcolor=color)

fig.suptitle("WGAN Loss Curves")
fig.tight_layout()
plt.show()

In [ ]:
# Generate samples from trained WGAN
wgan_generator.eval()
with torch.no_grad():
    noise = torch.randn(25, 100).to(device)
    samples = wgan_generator(noise).cpu().numpy()
plot_generated_images(samples, 'Final', title='WGAN Generated Samples')